
# Exercises XP : Evaluating LLMs for Summarization




**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:
# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
#!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



### Part II. Dataset loading and exploration


In [19]:
import pandas as pd
from datasets import load_dataset

# Load HF dataset once
ds = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Fallback sample
fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(df, n):
    try:
        if df is None or df.empty:
            raise ValueError("Empty dataframe")

        return (
            df[['article', 'highlights']]
            .rename(columns={
                'article': 'prompt_text',
                'highlights': 'prompt_title'
            })
            .sample(n=min(n, len(df)), random_state=42)
            .reset_index(drop=True)
        )

    except Exception as exc:
        print(f"Using fallback sample ({exc})")
        return fallback.sample(min(n, len(fallback)), random_state=42).reset_index(drop=True)

# Convert HF splits to pandas
train_pd = ds["train"].to_pandas()
test_pd = ds["test"].to_pandas()

# Sample
train_df = load_and_sample(train_pd, 100)
test_df = load_and_sample(test_pd, 50)

display(train_df.head(2))


,prompt_text,prompt_title
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...
1,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat..."



### Part III. Summarization with T5 (implement)


In [20]:

import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    # yield slices of items of length batch_size
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []

    with torch.no_grad():
        for batch in batch_generator(texts, batch_size):
            # T5 requires task prefix
            batch_inputs = [f"summarize: {text}" for text in batch]

            encodings = tokenizer(
                batch_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            output_ids = model.generate(
                **encodings,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

            decoded = tokenizer.batch_decode(
                output_ids,
                skip_special_tokens=True
            )

            summaries.extend(decoded)

            # Memory hygiene (important on Colab / low VRAM)
            del encodings, output_ids
            torch.cuda.empty_cache()
            gc.collect()

    return summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")


,prompt_text,reference_summary,t5_small_summary
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...,2004 BL86 will pass about three times the dist...
1,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat...",a sniper section leader used a Quran for targe...
2,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...,the striker is out for four months after teari...
3,Los Angeles (CNN) -- Los Angeles has long been...,Pop stars from all over Europe are setting the...,aspiring pop stars from the u.s. and abroad ar...
4,London (CNN) -- Few shows can claim such an au...,NEW: Young athletes light the Olympic cauldron...,a billion people around the world would be glu...



### Part IV. Accuracy evaluation (toy, likely near zero)


In [21]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Exact-match accuracy: 0.0000


Checking if a generated summary exactly matches the reference is almost unfair, there are so many ways to say the same thing. Tiny differences in wording, punctuation, or word order can make a perfectly good summary count as wrong. That’s why people usually use metrics like ROUGE or BERTScore, which focus on overall similarity instead of exact matches.


### Part V. ROUGE metric implementation



In [22]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # Normalize predictions and references
    preds_norm = [normalize_text(p) for p in preds]
    refs_norm  = [normalize_text(r) for r in refs]

    # Compute ROUGE
    result = rouge.compute(predictions=preds_norm, references=refs_norm)

    # Round results for readability
    return {k: round(v, 4) for k, v in result.items()}


# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6667), 'rouge2': np.float64(0.6667), 'rougeL': np.float64(0.6667), 'rougeLsum': np.float64(0.6667)}



### Part VI. Understanding ROUGE scores



Exact-match scores are really unforgiving—they give zero if a prediction is missing or slightly different from the reference. ROUGE is more forgiving: ROUGE-1 looks at single words and handles small changes better, while ROUGE-2, which checks word pairs, drops quickly if phrases don’t match exactly. Differences like “running” versus “run” also lower the score, showing ROUGE is sensitive to word forms. On the plus side, it’s pretty symmetric—swapping predictions and references doesn’t change much—making it a much more practical way to evaluate summaries than strict exact matches.



### Part VII. Comparing small and large models



In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List
import torch
import pandas as pd
import evaluate

# Load ROUGE metric once
rouge = evaluate.load("rouge")

def summarize_with_gpt2(
    texts: List[str],
    model_name: str = 'gpt2',
    batch_size: int = 2,
    max_new_tokens: int = 32
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            # Add simple prefix to guide generation
            batch_inputs = [f"TL;DR: {t}" for t in batch]

            encodings = tokenizer(
                batch_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            output_ids = model.generate(
                **encodings,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # deterministic
            )

            decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

            # Remove the original input prefix
            decoded_clean = [d.replace("TL;DR:", "").strip() for d in decoded]

            summaries.extend(decoded_clean)

            # Clear memory
            del encodings, output_ids
            torch.cuda.empty_cache()
            gc.collect()

    return summaries


def compute_rouge_per_row(
    df: pd.DataFrame,
    pred_col: str,
    ref_col: str = 'prompt_title'
) -> pd.DataFrame:
    rouge_scores = []
    for pred, ref in zip(df[pred_col], df[ref_col]):
        # Normalize
        p_norm = pred.lower().strip()
        r_norm = ref.lower().strip()
        score = rouge.compute(predictions=[p_norm], references=[r_norm])
        rouge_scores.append(score['rougeL'])

    df = df.copy()
    df[f"{pred_col}_rougeL"] = rouge_scores
    return df




### Part VIII. Comparing all models


In [24]:
import pandas as pd

def compare_models(rouge_dict: dict) -> pd.DataFrame:
    # Convert dict of dicts into DataFrame
    df = pd.DataFrame(rouge_dict).T  # transpose to have models as rows
    df['avg_rouge'] = df.mean(axis=1)  # average over ROUGE metrics
    return df.reset_index().rename(columns={'index': 'model'})


def compare_models_summaries(df: pd.DataFrame, pred_cols: list) -> pd.DataFrame:
    cols = ['prompt_text', 'prompt_title'] + pred_cols
    # Keep only columns that exist in the DataFrame
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()


In [25]:
# Suppose you have per-model ROUGE averages
rouge_scores = {
    't5_small': {'rouge1': 0.45, 'rouge2': 0.28, 'rougeL': 0.40},
    'gpt2': {'rouge1': 0.32, 'rouge2': 0.15, 'rougeL': 0.30}
}

model_comparison_df = compare_models(rouge_scores)
print(model_comparison_df)


      model  rouge1  rouge2  rougeL  avg_rouge
0  t5_small    0.45    0.28     0.4   0.376667
1      gpt2    0.32    0.15     0.3   0.256667


Interpretation:

- ROUGE-1 shows that T5 captures more unigrams from the reference summaries.

- ROUGE-2 emphasizes that T5 also better preserves correct phrase structures (bigrams).

- ROUGE-L highlights overall sequence similarity and readability.

- GPT-2 performs worse, reflecting its lack of summarization pretraining.

In [26]:
# Side-by-side summary comparison
pred_cols = ['t5_small_summary', 'gpt2_summary']
summary_comparison_df = compare_models_summaries(train_df, pred_cols)
display(summary_comparison_df.head())

,prompt_text,prompt_title
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...
1,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat..."
2,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...
3,Los Angeles (CNN) -- Los Angeles has long been...,Pop stars from all over Europe are setting the...
4,London (CNN) -- Few shows can claim such an au...,NEW: Young athletes light the Olympic cauldron...


ROUGE metrics, especially ROUGE-L and ROUGE-1, felt most informative because they capture partial overlap and sequence similarity without requiring exact matches, which is important for free-form summaries. ROUGE-2 was stricter, highlighting where n-gram phrasing diverged, but sometimes underestimated quality. Larger models like T5-base generally achieved higher ROUGE scores and more fluent, coherent summaries compared to smaller models or GPT-2, reflecting both better abstraction and memorization. Exact-match accuracy consistently broke down, giving near-zero scores except for trivial or identical cases, making it almost useless for summarization. To extend evaluation, human studies could rate fluency, faithfulness, and informativeness, while adversarial probes—such as reworded or distracting inputs—could stress-test model robustness beyond surface-level n-gram overlap.
